<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Transfer learning

**[Transfer learning](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/)** · Vision AI: See, Count and Inspect · Module 2, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** use a model that learned from ImageNet's photos for Larkfield's catalogue: first by training only a new last layer, then by fine-tuning its last block, with and without augmentation. You add the rows to the comparison table with their CPU time.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells and two models (47 MB and 15 MB). No account and no key. |
| **You should know** | What a convolution filter does, from [Convolution and patches, in one lesson](https://learning.nextia-ai.com/courses/vision/m02/convolution-and-patches-in-one-lesson/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M02-L04-transfer-learning/transfer-learning-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 15 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A pretrained model, as it is

ResNet-18 learned from about 1.3 million ImageNet photos in 1,000 classes. The next cell downloads it (47 MB) once, and asks it about three catalogue photos with its own classes.

> **Predict.** ImageNet has no class "lamp". What will the model say about a carved pumpkin with a light inside?

In [ ]:
import time
import torch, timm
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)

# Pinned weights (timm, Apache-2.0 weights trained on ImageNet-1k), so that everyone gets the same numbers.
WEIGHTS = {"resnet18.a1_in1k": "491b427b45c94c7fb0e78b5474cc919aff584bbf",
           "convnext_atto.d2_in1k": "435c0a8d02d5b872e90c8cde36887465982cb399"}

def load(name, **kw):
    f = hf_hub_download("timm/" + name, "model.safetensors", revision=WEIGHTS[name])
    return timm.create_model(name, pretrained=True, pretrained_cfg_overlay={"file": f}, **kw)

def embed(model, paths, batch=32):
    """The model without its last layer: one list of numbers per photo, scaled to length 1."""
    config = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**config)
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), batch):
            x = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths[i:i + batch]])
            out.append(model(x).numpy())
    X = np.vstack(out)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
studio = pd.read_csv("data/catalogue/new_studio_manifest.csv")

full = load("resnet18.a1_in1k").eval()
config = timm.data.resolve_data_config({}, model=full)
transform = timm.data.create_transform(**config)
names = timm.data.ImageNetInfo()
for f in ["test/mug/P-ab240c16.jpg", "test/lamp/P-b87b9773.jpg", "test/cushion/P-a4e99b4e.jpg"]:
    with torch.no_grad():
        p = full(transform(Image.open("data/catalogue/" + f).convert("RGB"))[None])[0].softmax(0)
    top = p.topk(2)
    print(f, "->", [(names.index_to_description(int(i)), round(float(v), 2)) for v, i in zip(top.values, top.indices)])

> **Check.** The mug is an "espresso" or a "cup", the pumpkin is a "jack-o'-lantern" and the cushion is a "studio couch". The model sees the photos well, but it answers with ImageNet's 1,000 classes, not Larkfield's 8.

## 2. Frozen: keep the model, train a new last layer

We remove the last layer (`num_classes=0`). What is left turns each photo into 512 numbers. We **freeze** the model: its weights do not change. Only a new last layer learns: a logistic regression on the 512 numbers.

In [ ]:
from sklearn.linear_model import LogisticRegression
backbone = load("resnet18.a1_in1k", num_classes=0).eval()
start = time.perf_counter()
Xtr = embed(backbone, list("data/" + train["file"]))
Xte = embed(backbone, list("data/" + test["file"]))
ms = 1000 * (time.perf_counter() - start) / (len(train) + len(test))
Xst = embed(backbone, list("data/" + studio["file"]))
head = LogisticRegression(C=1.0, max_iter=5000).fit(Xtr, train["label"])
for name, X, y in (("catalogue test", Xte, test["label"]), ("new studio", Xst, studio["label"])):
    s = cp.classification_scores(list(y), list(head.predict(X)), CLASSES)
    print(f"{name:15s} accuracy {s['accuracy']}  worst {s['worst_class']} {s['worst_recall']}")
print(f"{ms:.1f} ms per photo on this computer")

> **Check.** You should see 0.812 on the test photos and 0.794 on the new studio. The histogram reached 0.263 and HOG 0.356 on the same photos. Your time per photo will differ.

## 3. Add the row to your comparison table

In [ ]:
mine = cp.Table()
mine.add(cp.classification_row("ResNet-18, frozen + last layer (yours)", list(test["label"]), list(head.predict(Xte)),
                               "catalogue test", round(ms, 1), 47, len(train), "Apache-2.0 (weights)"))
mine.show("catalogue", columns=("method", "value", "worst", "worst_value", "ms_per_image", "model_mb"))

## 4. Fine-tune: let the last block learn too

Now the last block of ResNet-18 (`layer4`) also learns, with the new last layer, for 4 passes over the 453 training photos (**epochs**). The rest stays frozen. The cell takes about 30 seconds on a laptop, and longer on a free Colab or Kaggle computer.

In [ ]:
from torchvision import transforms as T
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
plain = T.Compose([T.Resize(224), T.ToTensor(), norm])
idx = {k: i for i, k in enumerate(CLASSES)}
images = {f: Image.open("data/" + f).convert("RGB") for f in list(train["file"]) + list(test["file"])}

def fine_tune(transform, epochs=4, seed=0):
    torch.manual_seed(seed)
    model = load("resnet18.a1_in1k", num_classes=len(CLASSES))   # a new last layer with 8 outputs
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("layer4") or name.startswith("fc")   # train the last block and the new layer
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
    y = torch.tensor([idx[k] for k in train["label"]])
    files = list(train["file"])
    g = torch.Generator().manual_seed(seed)
    start = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(files), generator=g), 0.0
        for i in range(0, len(files), 32):
            b = order[i:i + 32]
            x = torch.stack([transform(images[files[j]]) for j in b.tolist()])
            loss = torch.nn.functional.cross_entropy(model(x), y[b])
            opt.zero_grad(); loss.backward(); opt.step()
            total += float(loss) * len(b)
        print(f"epoch {epoch + 1}: loss {total / len(files):.3f}")
    print(f"training: {time.perf_counter() - start:.0f} s")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(test), 32):
            x = torch.stack([plain(images[f]) for f in test["file"][i:i + 32]])
            preds += [CLASSES[k] for k in model(x).argmax(1).tolist()]
    return cp.classification_scores(list(test["label"]), preds, CLASSES)

s = fine_tune(plain)
print("fine-tuned, no augmentation:", s["accuracy"], "| worst", s["worst_class"], s["worst_recall"])

> **Check.** The loss falls in each epoch. The course team measured 0.85 for this run. On another computer, the last digit can differ a little, because the order of the calculations changes.

## 5. Augmentation: new copies of the photos

**Augmentation** changes each training photo a little in every epoch: a mirror image, a smaller crop, other colours. The model sees more kinds of photos. Look at what it does to one photo:

In [ ]:
augment = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                     T.ColorJitter(0.4, 0.4, 0.4, 0.08)])
torch.manual_seed(1)
one = images[train["file"][0]]
fig, axes = plt.subplots(1, 6, figsize=(13, 2.4))
for i, ax in enumerate(axes):
    ax.imshow(one if i == 0 else augment(one)); ax.axis("off")
    ax.set_title("original" if i == 0 else "changed by code", fontsize=8)
plt.show()

The course team fine-tuned the same model with two kinds of augmentation (the same code, `fine_tune(transform)`). Flips and crops gave 0.844; flips, crops and colour gave 0.863; no augmentation gave 0.85. With 160 test photos, one photo is 0.6 points: these differences are noise. Augmentation did not help here. You can run `fine_tune(T.Compose([augment, T.ToTensor(), norm]))` yourself: it takes about 30 seconds more.

## 6. Your turn: a smaller model

ConvNeXt-Atto is a newer design and much smaller: 15 MB against 47 MB for ResNet-18.

> **Your turn.** Load `"convnext_atto.d2_in1k"` with `num_classes=0`, embed the training and test photos with `embed`, train the same `LogisticRegression(C=1.0, max_iter=5000)`, and put the test accuracy in `acc_convnext` (as `cp.classification_scores` gives it). Time the embedding of the train and test photos, and put the milliseconds per photo in `ms_convnext`. Then add your row to `mine`.

In [ ]:
acc_convnext = ...
ms_convnext = ...

In [ ]:
expect('acc_convnext', acc_convnext, 0.887, tolerance=0.01)

Is the smaller model worse? Compare its row with ResNet-18's, and with the course's ResNet-50 row (0.894, 103 MB) in `cp.Table.load("data/comparison_table.json")`. Write two sentences for Priya's notes: which model would you choose for the catalogue, and why?

## Next

You added pretrained models to the comparison table. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.